# Install LM Evaluation Harness Packages

In [1]:
!git clone --depth 1 https://github.com/EleutherAI/lm-evaluation-harness

fatal: destination path 'lm-evaluation-harness' already exists and is not an empty directory.


In [3]:
%%writefile /kaggle/working/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py
from typing import Dict, List

import datasets


def process_docs(dataset: datasets.Dataset) -> datasets.Dataset:
    def _process_doc(doc: dict) -> dict:
        out_doc = {
            "problem": doc["problem"],
            "solution": doc["solution"],
            "answer": remove_boxed(last_boxed_only_string(doc["solution"])),
        }
        return out_doc

    return dataset.map(_process_doc)


def process_results(doc: dict, results: List[str]) -> Dict[str, int]:
    retval = 0
    response = results[0]

    # Try to extract \boxed{} from the response (matches aime task behavior)
    boxed = last_boxed_only_string(response)
    if boxed is not None:
        try:
            answer = remove_boxed(boxed)
        except AssertionError:
            answer = None
    else:
        answer = None

    # Fall back to $...$ extraction
    if answer is None:
        indices = [pos for pos, char in enumerate(response) if char == "$"]
        if len(indices) <= 1:
            answer = response
        else:
            answer = response[indices[0] + 1 : indices[-1]]

    if is_equiv(answer, remove_boxed(last_boxed_only_string(doc["solution"]))):
        retval = 1

    results = {
        "exact_match": retval,
    }
    return results


# string normalization from https://github.com/EleutherAI/lm-evaluation-harness/blob/master/lm_eval/tasks/hendrycks_math.py
def is_equiv(str1, str2, verbose=False):
    if str1 is None and str2 is None:
        print("WARNING: Both None")
        return True
    if str1 is None or str2 is None:
        return False

    try:
        ss1 = strip_string(str1)
        ss2 = strip_string(str2)
        if verbose:
            print(ss1, ss2)
        return ss1 == ss2
    except Exception:
        return str1 == str2


def remove_boxed(s):
    if "\\boxed " in s:
        left = "\\boxed "
        assert s[: len(left)] == left
        return s[len(left) :]

    left = "\\boxed{"

    assert s[: len(left)] == left
    assert s[-1] == "}"

    return s[len(left) : -1]


def last_boxed_only_string(string):
    idx = string.rfind("\\boxed")
    if "\\boxed " in string:
        return "\\boxed " + string.split("\\boxed ")[-1].split("$")[0]
    if idx < 0:
        idx = string.rfind("\\fbox")
        if idx < 0:
            return None

    i = idx
    right_brace_idx = None
    num_left_braces_open = 0
    while i < len(string):
        if string[i] == "{":
            num_left_braces_open += 1
        if string[i] == "}":
            num_left_braces_open -= 1
            if num_left_braces_open == 0:
                right_brace_idx = i
                break
        i += 1

    if right_brace_idx is None:
        retval = None
    else:
        retval = string[idx : right_brace_idx + 1]

    return retval


def fix_fracs(string):
    substrs = string.split("\\frac")
    new_str = substrs[0]
    if len(substrs) > 1:
        substrs = substrs[1:]
        for substr in substrs:
            new_str += "\\frac"
            if substr[0] == "{":
                new_str += substr
            else:
                try:
                    assert len(substr) >= 2
                except AssertionError:
                    return string
                a = substr[0]
                b = substr[1]
                if b != "{":
                    if len(substr) > 2:
                        post_substr = substr[2:]
                        new_str += "{" + a + "}{" + b + "}" + post_substr
                    else:
                        new_str += "{" + a + "}{" + b + "}"
                else:
                    if len(substr) > 2:
                        post_substr = substr[2:]
                        new_str += "{" + a + "}" + b + post_substr
                    else:
                        new_str += "{" + a + "}" + b
    string = new_str
    return string


def fix_a_slash_b(string):
    if len(string.split("/")) != 2:
        return string
    a = string.split("/")[0]
    b = string.split("/")[1]
    try:
        a = int(a)
        b = int(b)
        assert string == "{}/{}".format(a, b)
        new_string = "\\frac{" + str(a) + "}{" + str(b) + "}"
        return new_string
    except AssertionError:
        return string


def remove_right_units(string):
    # "\\text{ " only ever occurs (at least in the val set) when describing units
    if "\\text{ " in string:
        splits = string.split("\\text{ ")
        assert len(splits) == 2
        return splits[0]
    else:
        return string


def fix_sqrt(string):
    if "\\sqrt" not in string:
        return string
    splits = string.split("\\sqrt")
    new_string = splits[0]
    for split in splits[1:]:
        if split[0] != "{":
            a = split[0]
            new_substr = "\\sqrt{" + a + "}" + split[1:]
        else:
            new_substr = "\\sqrt" + split
        new_string += new_substr
    return new_string


def strip_string(string):
    # linebreaks
    string = string.replace("\n", "")

    # remove inverse spaces
    string = string.replace("\\!", "")

    # replace \\ with \
    string = string.replace("\\\\", "\\")

    # replace tfrac and dfrac with frac
    string = string.replace("tfrac", "frac")
    string = string.replace("dfrac", "frac")

    # remove \left and \right
    string = string.replace("\\left", "")
    string = string.replace("\\right", "")

    # Remove circ (degrees)
    string = string.replace("^{\\circ}", "")
    string = string.replace("^\\circ", "")

    # remove dollar signs
    string = string.replace("\\$", "")

    # remove units (on the right)
    string = remove_right_units(string)

    # remove percentage
    string = string.replace("\\%", "")
    string = string.replace("\%", "")  # noqa: W605

    # " 0." equivalent to " ." and "{0." equivalent to "{." Alternatively, add "0" if "." is the start of the string
    string = string.replace(" .", " 0.")
    string = string.replace("{.", "{0.")
    # if empty, return empty string
    if len(string) == 0:
        return string
    if string[0] == ".":
        string = "0" + string

    # to consider: get rid of e.g. "k = " or "q = " at beginning
    if len(string.split("=")) == 2:
        if len(string.split("=")[0]) <= 2:
            string = string.split("=")[1]

    # fix sqrt3 --> sqrt{3}
    string = fix_sqrt(string)

    # remove spaces
    string = string.replace(" ", "")

    # \frac1b or \frac12 --> \frac{1}{b} and \frac{1}{2}, etc. Even works with \frac1{72} (but not \frac{72}1). Also does a/b --> \\frac{a}{b}
    string = fix_fracs(string)

    # manually change 0.5 --> \frac{1}{2}
    if string == "0.5":
        string = "\\frac{1}{2}"

    # NOTE: X/Y changed to \frac{X}{Y} in dataset, but in simple cases fix in case the model output is X/Y
    string = fix_a_slash_b(string)

    return string

Overwriting /kaggle/working/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py


In [2]:
%cd lm-evaluation-harness
!pip install -e .

/kaggle/working/lm-evaluation-harness
Obtaining file:///kaggle/working/lm-evaluation-harness
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 6.9 MB/s eta 0:00:00
  Building editable for lm_eval (pyproject.toml) ... done
  Created wheel for lm_eval: filename=lm_eval-0.4.14.dev0-0.editable-py3-none-any.whl size=32256 sha256=972f7a380d5063711cc7882ad75502ea9b1a684db643a87e520e3090c65a4e70
  Stored in directory: /tmp/pip-ephem-wheel-cache-6ey3d8fd/wheels/a2/f4/5c

# Run Evaluations Qwen 3.4

## **persian GRPO**

In [4]:
!accelerate launch -m lm_eval --model hf \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --model_args pretrained=arefehRajabian/Qwen3-4B-Base-persian-math-grpo \
    --gen_kwargs max_new_tokens=1024 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/results/qwen3_1.7b \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Use standard mathematical notation for fractions, radicals, equations, and other expressions. Put your final answer in \boxed{...}."

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `2`
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-17:12:01:21 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-17:12:01:21 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-17:12:01:32 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_

## **SFT**

In [4]:
!accelerate launch -m lm_eval --model hf \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --model_args pretrained=arefehRajabian/Qwen3_4B_baase_SFT_16bit \
    --gen_kwargs max_new_tokens=1024 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/results/qwen3_1.7b \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Use standard mathematical notation for fractions, radicals, equations, and other expressions. Put your final answer in \boxed{...}."

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `2`
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-21:18:47:08 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-21:18:47:08 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-21:18:47:19 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_

## **base**

In [4]:
!accelerate launch -m lm_eval --model hf \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --model_args pretrained=unsloth/Qwen3-4B-Base \
    --gen_kwargs max_new_tokens=1024 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/results/qwen \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Use standard mathematical notation for fractions, radicals, equations, and other expressions. Put your final answer in \boxed{...}."

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `2`
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-22:13:51:04 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-22:13:51:04 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-22:13:51:16 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_

In [5]:
!accelerate launch -m lm_eval --model hf \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --model_args pretrained=arefehRajabian/Qwen3-4B-Base-english-math-grpo \
    --gen_kwargs max_new_tokens=1024 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/results/qwen \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Use standard mathematical notation for fractions, radicals, equations, and other expressions. Put your final answer in \boxed{...}."

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `2`
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-22:17:45:13 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-22:17:45:13 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-22:17:45:24 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_